# 01 — Data Processing

Loads PADS raw data, filters to RightWrist + Relaxed task, applies bandpass filter,
segments into windows, and saves the four files that all downstream notebooks depend on:

```
data/processed/windows.npy       — shape (N, 6, 200)
data/processed/labels.npy        — shape (N,)  0=HC  1=PD
data/processed/subject_ids.npy   — shape (N,)  integer subject ID per window
data/processed/fold_splits.pkl   — 5-fold CV subject-level splits
```

Run this once. All other notebooks load from these files.

## 0. Setup

In [ ]:
import sys
import os
import os.path as osp
import json
import zipfile
import numpy as np
import pandas as pd

# Add repo root to path so src/ imports work
sys.path.append(os.path.abspath('..'))

from src.data.preprocessing import bandpass_filter, segment_windows
from src.data.folds import generate_fold_splits

# --- Paths — edit these for your environment ---
# Kaggle:  ZIP_PATH = '/kaggle/input/pads-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0.zip'
# Colab:   ZIP_PATH = '/content/drive/MyDrive/pads-parkisons-dataset-folder/pads-parkinsons-disease-smartwatch-dataset-1.0.0.zip'
ZIP_PATH     = '/kaggle/input/pads-dataset/pads-parkinsons-disease-smartwatch-dataset-1.0.0.zip'
EXTRACT_PATH = '/tmp/pads_dataset'
OUTPUT_DIR   = '../data/processed'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## 1. Extract dataset

In [ ]:
if not osp.exists(EXTRACT_PATH):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print("Done.")
else:
    print("Already extracted.")

BASE_PATH       = osp.join(EXTRACT_PATH, 'pads-parkinsons-disease-smartwatch-dataset-1.0.0')
TIMESERIES_DIR  = osp.join(BASE_PATH, 'movement', 'timeseries')
OBSERVATION_DIR = osp.join(BASE_PATH, 'movement')
PATIENTS_DIR    = osp.join(BASE_PATH, 'patients')

print(os.listdir(BASE_PATH))

## 2. Build manifest

In [ ]:
def get_patient_label(patient_id, patients_dir):
    individual_file = f"patient_{int(patient_id):03d}.json"
    file_path = osp.join(patients_dir, individual_file)
    with open(file_path) as f:
        contents = json.load(f)
    return contents.get('condition')


def build_manifest(patients_dir, observation_dir, n_patients=469):
    rows = []
    for pat_num in range(1, n_patients + 1):
        patient_id = f"{pat_num:03d}"
        label = get_patient_label(pat_num, patients_dir)

        obs_path = osp.join(observation_dir, f"observation_{patient_id}.json")
        with open(obs_path) as f:
            obs_contents = json.load(f)

        for session in obs_contents['session']:
            activity_name = session.get('record_name')
            for record in session['records']:
                wrist    = record.get('device_location')
                filename = record.get('file_name')
                rows.append({
                    'patient_id': patient_id,
                    'label':      label,
                    'task':       activity_name,
                    'wrist':      wrist,
                    'filepath':   filename,
                })
    return pd.DataFrame(rows)


manifest = build_manifest(PATIENTS_DIR, OBSERVATION_DIR, n_patients=469)
print(f"Total manifest rows: {len(manifest)}")
print(manifest['label'].value_counts())
print(manifest['task'].value_counts())
print(manifest['wrist'].value_counts())

## 3. Filter to RightWrist + Relaxed task + HC/PD only

In [ ]:
filtered = manifest[
    (manifest['task']  == 'Relaxed') &
    (manifest['wrist'] == 'RightWrist') &
    (manifest['label'].isin(['HC', 'PD']))
].copy()

print(f"Filtered rows: {len(filtered)}")
print(filtered['label'].value_counts())
print(f"Unique subjects: {filtered['patient_id'].nunique()}")

## 4. Load timeseries, bandpass filter, segment into windows

In [ ]:
def load_timeseries(filepath):
    """Load a PADS timeseries CSV. Returns numpy array (6, n_samples)."""
    df = pd.read_csv(filepath, header=None)
    # Columns: Time, Accel_X, Accel_Y, Accel_Z, Gyro_X, Gyro_Y, Gyro_Z
    # Drop Time column, keep 6 sensor channels
    signal = df.iloc[:, 1:7].values.T  # shape (6, n_samples)
    return signal.astype(np.float32)


# --- Build windows, labels, subject_ids ---
all_windows     = []
all_labels      = []
all_subject_ids = []

LABEL_MAP = {'HC': 0, 'PD': 1}
FS          = 100.0
WINDOW_SIZE = 200   # 2 seconds at 100Hz
LOWCUT      = 1.0
HIGHCUT     = 20.0
N_WINDOWS   = 10    # keep exactly 10 windows per subject

skipped = 0

for _, row in filtered.iterrows():
    filepath = osp.join(BASE_PATH, 'movement', row['filepath'])

    if not osp.exists(filepath):
        skipped += 1
        continue

    try:
        signal = load_timeseries(filepath)  # (6, n_samples)
    except Exception:
        skipped += 1
        continue

    # Must have enough samples for at least N_WINDOWS windows
    if signal.shape[1] < WINDOW_SIZE * N_WINDOWS:
        skipped += 1
        continue

    # Bandpass filter
    filtered_signal = bandpass_filter(signal, lowcut=LOWCUT,
                                       highcut=HIGHCUT, fs=FS)

    # Segment into non-overlapping windows
    windows = segment_windows(filtered_signal,
                               window_size=WINDOW_SIZE,
                               step=WINDOW_SIZE)  # (n_windows, 6, 200)

    # Keep exactly N_WINDOWS
    windows = windows[:N_WINDOWS]
    if len(windows) < N_WINDOWS:
        skipped += 1
        continue

    label      = LABEL_MAP[row['label']]
    subject_id = int(row['patient_id'])

    all_windows.append(windows)
    all_labels.extend([label] * N_WINDOWS)
    all_subject_ids.extend([subject_id] * N_WINDOWS)

print(f"Skipped: {skipped} subjects")
print(f"Processed: {len(all_windows)} subjects")

## 5. Stack and verify

In [ ]:
windows     = np.concatenate(all_windows, axis=0)  # (N, 6, 200)
labels      = np.array(all_labels,      dtype=np.int64)
subject_ids = np.array(all_subject_ids, dtype=np.int64)

print(f"Windows:     {windows.shape}")
print(f"Labels:      {labels.shape}  — HC: {(labels==0).sum()}  PD: {(labels==1).sum()}")
print(f"Subject IDs: {subject_ids.shape}  — Unique: {len(np.unique(subject_ids))}")

assert windows.shape[1] == 6
assert windows.shape[2] == 200
assert len(windows) == len(labels) == len(subject_ids)
print("All checks passed.")

## 6. Generate fold splits

In [ ]:
# One label per subject — take label from first window of each subject
unique_subjects = np.unique(subject_ids)
subject_labels  = np.array([
    labels[subject_ids == s][0] for s in unique_subjects
])

folds = generate_fold_splits(
    subject_ids=unique_subjects,
    subject_labels=subject_labels,
    n_splits=5,
    random_state=42,
    val_fraction=0.2,
    save_path=osp.join(OUTPUT_DIR, 'fold_splits.pkl'),
)

# Verify fold sizes
for i, fold in enumerate(folds):
    print(f"Fold {i+1}: "
          f"train={len(fold['train_subjects'])} "
          f"val={len(fold['val_subjects'])} "
          f"test={len(fold['test_subjects'])} subjects")

## 7. Save arrays

In [ ]:
np.save(osp.join(OUTPUT_DIR, 'windows.npy'),     windows)
np.save(osp.join(OUTPUT_DIR, 'labels.npy'),      labels)
np.save(osp.join(OUTPUT_DIR, 'subject_ids.npy'), subject_ids)

print("Saved:")
for fname in ['windows.npy', 'labels.npy', 'subject_ids.npy', 'fold_splits.pkl']:
    path = osp.join(OUTPUT_DIR, fname)
    size = osp.getsize(path) / 1e6
    print(f"  {fname}  ({size:.1f} MB)")

print("\nData processing complete. Ready for notebook 03.")